# Banking Customer Support AI Agent — Multi-Agent Architecture
### Applied Generative AI · Capstone Project

A **multi-agent GenAI system** for banking customer support. A **Classifier Agent**
reads each customer message and routes it to the right specialist agent:

- **Positive Feedback** → *Feedback Handler* returns a warm, personalized thank-you (LLM).
- **Negative Feedback** → *Feedback Handler* creates a new 6-digit ticket in the support
  database and returns an empathetic acknowledgement.
- **Query** → *Query Handler* extracts the ticket number and returns its status from the DB.

| Capstone step | Where |
|---|---|
| 1. Classifier Agent | §2 |
| 2. Feedback Handler Agent | §3 |
| 3. Query Handler Agent | §4 |
| 4. Agent coordination (sample flows) | §5 |
| 7. Model evaluation | §6 |
| 8. Streamlit UI | §8 |
| 9. Logs & debugging | §7 |

> The engine lives in **`support_agents_core.py`**; this notebook imports it, runs every
> step, and explains the design. It runs end-to-end even without a key (using a
> rule-based fallback); with an `OPENAI_API_KEY` the LLM agents are active.

## 0. Setup
Install dependencies (`pip install -r requirements.txt`) and put your key in a `.env`
file (`OPENAI_API_KEY=sk-...`). Keys are read via `os.getenv` and never hard-coded.

In [10]:
import os
from dotenv import load_dotenv
load_dotenv()
HAS_KEY = bool(os.getenv("OPENAI_API_KEY"))
print("OpenAI key detected:", HAS_KEY,
      "(LLM agents active)" if HAS_KEY else "(using rule-based fallback)")

OpenAI key detected: True (LLM agents active)


## 1. The support database
Negative feedback creates tickets, and queries look them up, so we start with a small
SQLite `support_tickets` table seeded with example tickets.

In [11]:
from support_agents_core import SupportDB
import pandas as pd

db = SupportDB("support.db"); db.seed()
pd.DataFrame(db.all_tickets())

,ticket_number,customer_name,issue,status,created_at
0,789866,Aman Malik,My debit card replacement still hasn't arrived.,Unresolved,2026-08-01T18:03:51.547692+00:00
1,196629,Customer,My debit card replacement still hasn't arrived.,Unresolved,2026-08-01T18:03:45.454511+00:00
2,591690,Customer,My debit card replacement still hasn't arrived...,Unresolved,2026-08-01T18:03:35.722262+00:00
3,892257,Frustrating,"I was charged twice for the same transaction, ...",Unresolved,2026-08-01T17:57:18.180007+00:00
4,733963,Aisha,I was charged twice for one transaction.,Unresolved,2026-08-01T17:57:04.036801+00:00
5,905612,David Lee,Loan statement not generated,In Progress,2026-07-30T10:05:00Z
6,120945,John Mathew,Incorrect interest charged,Unresolved,2026-07-28T11:20:00Z
7,784521,Aisha Khan,Debit card replacement not received,In Progress,2026-07-25T14:03:00Z
8,650932,Rahul Verma,Net banking login not working,Resolved,2026-07-20T09:12:00Z
9,330011,Neha Gupta,UPI transaction failed but debited,Resolved,2026-07-18T16:45:00Z


## 2. Classifier Agent  *(step 1)*
The Classifier Agent categorizes each message into **Positive Feedback**, **Negative
Feedback**, or **Query**. With a key it uses a zero-shot LLM prompt; without one it falls
back to a keyword heuristic. Its label decides the routing.

In [12]:
from support_agents_core import ClassifierAgent, get_llm

classifier = ClassifierAgent(get_llm())
for m in ["Thanks for resolving my issue so fast!",
          "My debit card still hasn't arrived.",
          "Could you check the status of ticket 650932?"]:
    print(f"{classifier.classify(m)['label']:18} <- {m}")

Positive Feedback  <- Thanks for resolving my issue so fast!
Query              <- My debit card still hasn't arrived.
Query              <- Could you check the status of ticket 650932?


## 3. Feedback Handler Agent  *(step 2)*
Activated for feedback. **Positive** → a personalized thank-you (LLM, with a template
fallback). **Negative** → generates a unique 6-digit ticket, inserts an *Unresolved*
row into `support_tickets`, and returns an empathetic message with the ticket number.

In [13]:
from support_agents_core import FeedbackHandlerAgent

fh = FeedbackHandlerAgent(db, get_llm())
print("POSITIVE:", fh.handle("Thank you, great service!", "Positive Feedback", "Rahul")["response"])
neg = fh.handle("I was charged twice for one transaction.", "Negative Feedback", "Aisha")
print("NEGATIVE:", neg["response"])
print("New ticket in DB:", db.get_ticket(neg["ticket_number"]))

POSITIVE: Hi Rahul,  

Thank you so much for your kind words! We're thrilled to hear you had a great experience, and we're always here to help.  

Best regards,  
[Your Name]
NEGATIVE: We apologize for the inconvenience, Aisha. A new ticket #804842 has been generated, and our team will follow up shortly.
New ticket in DB: {'ticket_number': '804842', 'customer_name': 'Aisha', 'issue': 'I was charged twice for one transaction.', 'status': 'Unresolved', 'created_at': '2026-08-01T18:15:41.513935+00:00'}


## 4. Query Handler Agent  *(step 3)*
Activated for queries. It extracts the ticket number from the message, looks it up in the
database, and returns the status — or asks for a valid number if none is found.

In [14]:
from support_agents_core import QueryHandlerAgent

qh = QueryHandlerAgent(db)
print(qh.handle("Could you check the status of ticket 650932?")["response"])
print(qh.handle("What's the update on 999999?")["response"])

Your ticket #650932 is currently marked as: Resolved.
I couldn't find ticket #999999 in our records. Please double-check the number.


## 5. Agent coordination — sample use-case flows  *(step 4)*
The **Orchestrator** ties the agents together: classify → route → respond → log. Below are
the three sample flows from the problem statement.

In [15]:
from support_agents_core import Orchestrator

orch = Orchestrator(db=db)   # uses the LLM automatically if a key is present
for msg in ["Thanks for sorting out my net banking login issue.",
            "My debit card replacement still hasn't arrived.",
            "Could you check the status of ticket 650932?"]:
    r = orch.process(msg)
    print(f"IN : {msg}")
    print(f"    → [{r['classification']}] {r['agent']}  ticket={r['ticket_number']}")
    print(f"    → {r['response']}\n")

IN : Thanks for sorting out my net banking login issue.
    → [Positive Feedback] Feedback Handler (Positive)  ticket=None
    → Dear Customer,  

Thank you so much for your kind words! I'm delighted to hear that I could help resolve your net banking login issue. If you need anything else, feel free to reach out anytime!  

Warm regards,  
[Your Name]

IN : My debit card replacement still hasn't arrived.
    → [Query] Query Handler  ticket=None
    → Could you please share your 6-digit ticket number so I can check its status?

IN : Could you check the status of ticket 650932?
    → [Query] Query Handler  ticket=650932
    → Your ticket #650932 is currently marked as: Resolved.



## 6. Model evaluation  *(step 7)*
We measure the **classification accuracy** and **agent routing success rate** over a
labelled test set (test-case coverage of the classifier logic).

In [16]:
from support_agents_core import evaluate_system

ev = evaluate_system(orch)
print(f"Classification accuracy: {ev['classification_accuracy']*100:.0f}%")
print(f"Routing success rate:    {ev['routing_success_rate']*100:.0f}%  "
      f"({ev['n_correct']}/{ev['n']})")
pd.DataFrame(ev["details"])

Classification accuracy: 88%
Routing success rate:    88%  (7/8)


,message,expected,predicted,correct
0,Thanks for sorting out my net banking login is...,Positive Feedback,Positive Feedback,True
1,I really appreciate how quickly you resolved m...,Positive Feedback,Positive Feedback,True
2,My debit card replacement still hasn't arrived.,Negative Feedback,Query,False
3,"I was charged twice for the same transaction, ...",Negative Feedback,Negative Feedback,True
4,Could you check the status of ticket 650932?,Query,Query,True
5,What is the update on ticket number 784521?,Query,Query,True
6,"Your service was excellent, thank you so much!",Positive Feedback,Positive Feedback,True
7,The mobile app keeps crashing when I try to pay.,Negative Feedback,Negative Feedback,True


In [17]:
# Optional: LLM-based QA scoring of response quality (empathy & clarity). Needs a key.
from support_agents_core import qa_score_responses
qa = qa_score_responses(orch)
print("Average response quality (1-5):", qa["avg_score"])
pd.DataFrame(qa["scored"])[["message","classification","score"]]

Average response quality (1-5): 3.4


,message,classification,score
0,Thanks for sorting out my net banking login is...,Positive Feedback,5
1,I really appreciate how quickly you resolved m...,Positive Feedback,5
2,My debit card replacement still hasn't arrived.,Query,2
3,"I was charged twice for the same transaction, ...",Negative Feedback,3
4,Could you check the status of ticket 650932?,Query,2


## 7. Logs & debugging view  *(step 9)*
Every interaction is logged: classification, agent, action, ticket, response, and success —
used for debugging and to compute the agent success rate.

In [18]:
print("Agent success rate:", orch.logger.success_rate())
pd.DataFrame(orch.logger.as_dicts())[["timestamp","classification","agent","action","ticket_number","success"]]

Agent success rate: 0.75


,timestamp,classification,agent,action,ticket_number,success
0,2026-08-01 18:15:43,Positive Feedback,Feedback Handler (Positive),thank_you,None,True
1,2026-08-01 18:15:43,Query,Query Handler,no_ticket_found,None,False
2,2026-08-01 18:15:44,Query,Query Handler,status_returned,650932,True
3,2026-08-01 18:15:50,Positive Feedback,Feedback Handler (Positive),thank_you,None,True
4,2026-08-01 18:15:52,Positive Feedback,Feedback Handler (Positive),thank_you,None,True
5,2026-08-01 18:15:53,Query,Query Handler,no_ticket_found,None,False
6,2026-08-01 18:15:54,Negative Feedback,Feedback Handler (Negative),ticket_created,141165,True
7,2026-08-01 18:15:55,Query,Query Handler,status_returned,650932,True


## 8. Streamlit UI  *(step 8)*
The interactive dashboard lives in **`banking_support_app.py`**. Run it from a terminal:

```bash
pip install -r requirements.txt
streamlit run banking_support_app.py
```

It opens at `http://localhost:8501` with four pages: **Support Assistant** (routing
simulator + test scenarios), **Support Tickets (DB)**, **Logs & Debug**, and **Evaluation**.

---
### Summary
This notebook implemented the full multi-agent banking-support system — a Classifier Agent
routing to a Feedback Handler (thank-you / ticket creation) and a Query Handler (status
lookup), coordinated by an Orchestrator, backed by a SQLite ticket database, with
evaluation, logging, and a Streamlit interface.